# Langextract aplication with atributes

In [25]:
import os
import re
import timeit
import random
import textwrap
import pandas as pd
import langextract as lx
from rich.pretty import pprint
from dotenv import load_dotenv
from more_itertools import unique_justseen
from aymurai.database.utils import text_to_uuid
from aymurai.api.endpoints.routers.misc.document_extract import extraction

In [ ]:
MAIN_DF ='df-NER-vals-02-08-sin05.csv'
DOCS_PATH = '/Users/sofi/Desktop/collectiveai/projects/AymurAI/backend/resources/data/sample/' #'/Users/sofi/Desktop/CollectiveAI/projects/data-genero/backend/resources/data/sample/' 

In [18]:
df = pd.read_csv(MAIN_DF)
df.head()

,text_x,NER_prediction,validation,id_x,created_at_x,updated_at,id_y,document_id,paragraph_id,order,created_at_y,name,paragraph_position,text_y,start_char,end_char,doc
0,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,[],[],c09fba61330852c4813fcbb2f3bec11e,2025-08-08 19:39:59-03:00,2025-08-08 19:40:45-03:00,57f85770879c420d8de2a0f3267b653b,518a7f34ad865b91ad95d954093f091a,c09fba61330852c4813fcbb2f3bec11e,NaN,2025-08-22 21:20:53-03:00,document-02.docx,1,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,148,233,document-02.docx
1,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,[],[],c09fba61330852c4813fcbb2f3bec11e,2025-08-08 19:39:59-03:00,2025-08-08 19:40:45-03:00,768db7897dc943489e9570a117975fe1,2536b5d2111d55c6b545e6bcbb75e002,c09fba61330852c4813fcbb2f3bec11e,NaN,2025-08-27 01:00:28-03:00,document-08.docx,1,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,62,147,document-08.docx
2,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,[],[],c09fba61330852c4813fcbb2f3bec11e,2025-08-08 19:39:59-03:00,2025-08-08 19:40:45-03:00,2c74793eac2341969aa3dcce7950f43b,6a7d2422da6a5852b68a7bee678d1aae,c09fba61330852c4813fcbb2f3bec11e,NaN,2025-08-27 02:56:14-03:00,document-04.docx,0,JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVEN...,0,85,document-04.docx
3,ANTECEDENTES,[],[],adee5f8144eb5e78abb3c56121e906cd,2025-08-08 19:40:02-03:00,2025-08-08 19:40:45-03:00,b31e11b3cff24069b7cf3faa666fbdec,0aec75365ad0511698f72bacb8b88212,adee5f8144eb5e78abb3c56121e906cd,NaN,2025-08-22 19:41:55-03:00,document-03.docx,9,ANTECEDENTES,857,869,document-03.docx
4,ANTECEDENTES,[],[],adee5f8144eb5e78abb3c56121e906cd,2025-08-08 19:40:02-03:00,2025-08-08 19:40:45-03:00,4abeb6f8ff3340cb99cfe61b16276012,518a7f34ad865b91ad95d954093f091a,adee5f8144eb5e78abb3c56121e906cd,NaN,2025-08-22 21:20:53-03:00,document-02.docx,9,ANTECEDENTES,725,737,document-02.docx


## Prompt & example definitions

In [26]:
PROMPT = textwrap.dedent("""
Sos un extractor de ENTIDADES sensibles y de ENTIDADES RELEVANTES para análisis en documentos judiciales en español. 
Leé cada párrafo y devolvé SOLO spans exactos (sin parafrasear ni inferir) de las clases definidas más abajo.

INSTRUCCIONES ESTRICTAS:
- Las iniciales de personas deben ser tomadas como PER (no confundir con siglas de otras cosas).
- Extraé SOLO spans EXACTOS que estén en el texto.
- Si una clase NO aparece, no devuelvas nada de esa clase.
- No inventes códigos ni números; no completes nada por contexto.
- No superpongas entidades; una mención = una extracción.
- No extraigas:
   1) La fecha y lugar de la resolución del documento (ejemplo: "Buenos Aires, 29 de julio de 2022"), que suele aparecer al comienzo.
   2) Información del juzgado como mail, dirección, teléfono o redes sociales (ejemplo: "Juzgado PCyF No 10 - Tacuarí 138, 7o Piso - juzcyf10@jusbaires.gob.ar - 4014-6821/20 - @jpcyf10"), que suele estar al pie del documento.
- Si una entidad está mal escrita, incompleta, repetida o en un formato no estándar, pero claramente corresponde, extraela igual.
- Si una entidad no está clara, no la extraigas.

AGRUPAMIENTO:
- Cada persona (PER) se agrupa con sus atributos en el mismo group_index: DNI, CUIT_CUIL, DIRECCION, TELEFONO, CORREO_ELECTRONICO, EDAD, NACIONALIDAD, NUM_MATRICULA.
- En PER podés usar atributo ROL_PROCESAL solo si está explícito en el texto. Valores posibles:
   "Juez", "Fiscal", "Secretario", "Prosecretario", "Mediador", "Asesor Tutelar", "Imputado", "Acusado", 
   "Victima", "Damnificado", "Denunciante", "Querellante", "Actor", "Demandado", "Testigo", 
   "Perito", "Defensor Oficial", "Defensor Particular", "Apoderado", "Tutor", "Curador", "Policia", 
   "Perjudicado", "Beneficiario".
- En DIRECCION podés usar atributo TIPO solo si está indicado en el texto. Valores posibles:
   "domicilio_real", "domicilio_legal", "domicilio_laboral", "domicilio_de_la_victima", "domicilio_del_imputado".
- Una DIRECCION puede expresarse como: 
   - calle + altura ("9 de Julio 123"),
   - intersección de calles ("Callao y Corrientes"),
   - número de domicilio aislado.

RELACIONES INTER-GRUPOS:
- Usá la clase RELACION para conectar grupos de personas.
- extraction_text = fragmento exacto de la medida o vínculo.
- attributes posibles:
   - TIPO: "prohibicion_acercamiento", "cese_perturbacion", "orden_cautelar", "contacto_prohibido", "restriccion_perimetral".
   - SUJETO_ACTIVO_GROUP: id del acusado/imputado.
   - SUJETO_PASIVO_GROUP: id de la víctima/denunciante/damnificado.
   - PLAZO: duración si está explícita (ejemplo: "6_meses").
   - DISTANCIA_MIN_M: distancia mínima en metros si está explícita (ejemplo: 500).
   - LUGARES_ALCANZADOS: direcciones o expresiones exactas como "cualquier lugar donde se encuentre la víctima".

CLASES:
- BANCO: entidad bancaria (no vale solo “Banco”).
- CBU: número de 22 dígitos.
- CORREO_ELECTRONICO: email (no el del juzgado).
- CUIT_CUIL: ##-########-#.
- CUIJ: código judicial ##-########-#.
- DIRECCION: domicilio en cualquiera de las formas listadas arriba.
- DNI: 7-8 dígitos o ##.###.### (no solo la palabra “DNI”).
- EDAD: edad en años o meses.
- ESTUDIOS: nivel educativo (primario, secundario, terciario, universitario, posgrado, doctorado; puede incluir “incompleto”, “completo”, “finalizado”, “en curso”).
- FECHA: fechas explícitas (excepto la de resolución del documento, y no expresiones vagas como “ayer” o “a las 15:00 horas”).
- LINK: URL.
- LOC: localidad, provincia, país, continente (no hospitales ni referencias genéricas).
- MARCA_AUTOMOVIL: marca de vehículo.
- NACIONALIDAD: nacionalidad.
- NUM_CAJA_AHORRO: número de caja de ahorro/cuenta.
- NUM_EXPEDIENTE: \d+/\d{4}.
- NUM_MATRICULA: matrícula profesional o académica.
- PATENTE_DOMINIO: dominio de vehículo (AAA123 o AA123AA).
- PER: nombre completo, iniciales o apodo de persona física.
- NUM_ACTUACION: número de actuación administrativa/contravencional.
- TELEFONO: fijo o celular.
- RELACION: vínculo o medida entre personas (ver atributos arriba).
""")


In [ ]:
NO_ATTRIBUTES_PROMPT = textwrap.dedent("""
Sos un extractor de ENTIDADES sensibles para anonimización en documentos judiciales en español. Vas a leer cada parrafo con atención y extraer todas las entidades que correspondan según
las clases definidas más abajo. 

INSTRUCCIONES ESTRICTAS:
- Las iniciales de personas deben ser tomadas como clase Persona, no confundir con siglas de otras cosas.
- Extraé SOLO spans EXACTOS que estén en el texto (no parafrasees ni infieras).
- Si una clase NO aparece, NO devuelvas nada de esa clase.
- NO inventes códigos ni números. No completes nada por contexto.
- No superpongas entidades; una mención = una extracción.
- NO son sensibles las siguientes entidades: 
                         1. La fecha y lugar de la resolucion del documento (por ej. "Buenos Aires, 29 de julio de 2022"), suele aparecen al comienzo del documento.
                         2. Información del juzgado como mail, direccion, telefono y cuenta de red social (por ej. "'Juzgado PCyF No 10 - Tacuarí 138, 7o Piso - juzcyf10@jusbaires.gob.ar - 4014-6821/20 - @jpcyf10'"), suele estar al pie del documento.
                         3. Las personas no sensibles como jueces, fiscales y secretarios.

- Si una entidad no está clara, NO la extraigas.
- Si una entidad está sutilmente mal escrita, incompleta, repetida o en un formato no estándar, pero detectas que corresponde a esa entendidad, extráela igual.

POSIBLES CLASES Y DESCRIPCIONES:
- BANCO: Debe especificar una entidad bancaria, 'Banco' no aplica como tal.
- CBU: número de 22 dígitos asociado a una entidad bancaria
- CORREO_ELECTRONICO: dirección de email, que no sea del juzgando.
- CUIT_CUIL: código único de identificación tributaria o laboral en Argentina (formato ##-########-#)
- CUIJ: código único de identificación judicial (formato ##-########-#)
- DIRECCION: puede presentarse como calle y altura (por ej. "9 de Julio 123"), intersección de calles (por ej. "Callao y Corrientes"), o número de domicilio 
- DNI: documento nacional de identidad de 7-8 dígitos, puede presentarse en el formato ##.###.### (numeros separados con puntos), la expresión "DNI" sin número no aplica como DNI.
- EDAD: edad de una persona, puede estar en años o meses
- ESTUDIOS: nivel educativo alcanzado (primario, secundario, terciario, universitario, posgrado, doctorado), puede estar acompañado de "incompleto", "completo", "finalizado", "en curso"
- FECHA: fechas en cualquier formato (dd/mm/aaaa, dd-mm-aaaa, dd de mes de aaaa, también puede ser dos fechas juntas como por ejemplo el 5 y 7 de mayo de 2020 y similares). No asignar como FECHA la fecha de resolución del documento, tampoco expresiones de fechas que no refieren a una fecha en particular, como por ejemplo, "en el día de ayer" o "a las 15:00 horas"
- LINK: URLs o enlaces web.
- LOC: nombres de localidades, provincias, países, continentes. NO asignar como LOC a lugares que no sean locaciones, como ser hospitales o referencias a lugares por nombres como "el domicilio de Olavarria"
- MARCA_AUTOMOVIL: marcas de automóviles (Ford, Chevrolet, Toyota, Renault, Fiat, etc)
- NACIONALIDAD: nacionalidades (argentina, italiana, española, uruguaya, chilena, paraguara, etc)
- NUM_CAJA_AHORRO: número de caja de ahorro o cuenta bancaria
- NUM_EXPEDIENTE: número de expediente judicial o administrativo en formato \d+/\d{4} (por ejemplo 1234/2020)
- NUM_MATRICULA: número de matrícula profesional (médica, abogacía, etc) o académica.
- PATENTE_DOMINIO: patentes o dominio de un vehículo. En Argentina, pueden ser de formato [A-Z]{3}\d{3} o [A-Z]{2}\d{3}[A-Z]{2}
- PER: Nombre y apellido(s) de una persona física. Los nombres inicializados y los apodos también cuentan como información sensible a anonimizar.                     
- NUM_ACTUACION: Número identificatorio de una actuación administrativa o contravencional.
- TELEFONO: Número telefónico (fijo o celular).

""")


In [ ]:
# -----------------
# Ejemplos balanceados (judiciales)
#   1) PER/FECHA/DIRECCION/LOC
#   2) Un único ejemplo de códigos (para enseñar formato)
#   3) Datos personales típicos de actuaciones
#   4) NEGATIVO: no hay códigos -> salida vacía
# -----------------
examples = [
    lx.data.ExampleData(
        text=textwrap.dedent("""En la Ciudad Autónoma de Buenos Aires, el día 5 de mayo de 2023, "
              "el Sr. Fiscal hace saber que Juan Pérez se domicilia en la calle "
              "Sarmiento 1234, localidad de Moreno."""),
        extractions=[
            lx.data.Extraction(extraction_class="FECHA",     extraction_text="5 de mayo de 2023"),
            lx.data.Extraction(extraction_class="PER",       extraction_text="Juan Pérez",attributes={'ROL':'Acusado'},group_index=1),
            lx.data.Extraction(extraction_class="DIRECCION", extraction_text="Sarmiento 1234",group_index=1),
            lx.data.Extraction(extraction_class="LOC",       extraction_text="Moreno",group_index=1),
        ],
    ),
lx.data.ExampleData(
    text=textwrap.dedent("""
        JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVENCIONAL Y DE FALTAS N°10 SECRETARIA N°19
        GOMEZ, ELVIS JUNIOR SOBRE 89 - LESIONES LEVES
        Número: IPP 8125/2020-0
        CUIJ: IPP J-01-00017381-5/2021-0
        Actuación Nro: 14544192/2021
        ACTA DE AUDIENCIA
        VIDEOCONFERENCIA
        "GOMEZ, ELVIS JUNIOR SOBRE 89 EN FUNCIÓN DEL 92, 149 BIS, 162 Y 239 DEL CÓDIGO PENAL"
        Causa N° 8122/2021
        Fecha: 4 de abril de 2021
        Horario de inicio: 12:00 horas
        Tipo de audiencia: audiencia de conocimiento personal (art. 266 CPPCABA)
        Juez: Pablo C. Casas -Juzgado Penal Contravencional y de Faltas Nro. 10-.
        Secretaria: Maria Agustina Iriarte López.
        PARTES PRESENTES
        Acusado: Carlos Junior PEREZ, DNI n° 50.966.533.
        Defensa Oficial: Marina Recabarra, -Defensoría Oficial Nro. 20-.
        Fiscal: Adrián Dávila -Fiscalía Penal, Contravencional y de Faltas Nro. 36-.
        DESARROLLO
        Juez: Da inicio a la audiencia...
        Fiscal: Explica que en virtud de la detención...
        El 15 de marzo de 2021, alrededor de las 23:00 horas, mientras se encontraba en una reunión en la casa de una señora llamada SOL,
        en Villa Pueyrredón de esta ciudad, se puso agresivo con su pareja BELEN GUTIERREZ, le pegó dos piñas en la cara,
        la agarró del cuello y la tiró al piso.
    """),
    extractions=[
        # Identificadores judiciales
        lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="8122/2021"),
        lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="8125/2020"),
        lx.data.Extraction(extraction_class="CUIJ", extraction_text="IPP J-01-00017381-5/2021-0"),
        lx.data.Extraction(extraction_class="NUM_ACTUACION", extraction_text="14544192/2021"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="4 de abril de 2021"),
        lx.data.Extraction(extraction_class="FECHA", extraction_text="15 de marzo de 2021"),
        lx.data.Extraction(extraction_class="LOC", extraction_text="Villa Pueyrredón"),

        # Funcionario judicial
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Pablo C. Casas",
            attributes={"ROL_PROCESAL": "Juez"},
            group_index=1
        ),
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Maria Agustina Iriarte López",
            attributes={"ROL_PROCESAL": "Secretario"},
            group_index=2
        ),
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Adrián Dávila",
            attributes={"ROL_PROCESAL": "Fiscal"},
            group_index=3
        ),
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Marina Recabarra",
            attributes={"ROL_PROCESAL": "Defensor Oficial"},
            group_index=4
        ),

        # Acusado
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Carlos Junior PEREZ",
            attributes={"ROL_PROCESAL": "Acusado"},
            group_index=5
        ),
        lx.data.Extraction(
            extraction_class="DNI",
            extraction_text="50.966.533",
            group_index=5
        ),

        # Víctima
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="BELEN GUTIERREZ",
            attributes={"ROL_PROCESAL": "Victima"},
            group_index=6
        ),

        # Testigo circunstancial (persona mencionada, no procesal)
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="SOL",
            group_index=7
        ),

        # Relación entre acusado y víctima (violencia física)
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="le pegó dos piñas en la cara, la agarró del cuello y la tiró al piso",
            attributes={
                "TIPO": "cese_perturbacion",   # violencia física → medida/restricción
                "SUJETO_ACTIVO_GROUP": 5,
                "SUJETO_PASIVO_GROUP": 6
            }
        )
    ],
),
lx.data.ExampleData(
    text=textwrap.dedent("""
        Hoy el Fiscal Carlos Garcia solicitó que se ordene a ROBERTO CARUZO, DNI 30.112.642, 34 años de edad, nacionalidad paraguaya, 
                con estudios secundarios completos, por el plazo de seis meses,
        el cese en los actos de perturbación o intimidación que, directa o indirectamente, realice hacia la persona de la
        damnificada BELEN CASIO, 37.412.987; 2) por el plazo de seis (6) meses, se imponga a ROBERTO CARUZO, DNI 30.112.642
        la prohibición de acercamiento y contacto hacia la víctima BELEN CASIO, 37.412.987, –en el lugar que se encuentre– de
        modo que deberá suspender todo tipo de contacto físico y/o por cualquier medio que signifique intromisión injustificada
        en relación a la persona de la damnificada, por sí o por intermedio de terceras personas y la prohibición de acercamiento
        a menos de 500 metros de los domicilios ubicados en Av. Pedro Goyena 51, piso 7° dpto. "A", y Callao 543, de esta Ciudad.
    """),
    extractions=[
        # Fiscal (grupo 0: funcionario, si querés diferenciarlo)
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="Carlos Garcia",
            attributes={"ROL_PROCESAL": "Fiscal"},
            group_index=0
        ),
        # Víctima
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="BELEN CASIO",
            attributes={"ROL_PROCESAL": "Victima"},
            group_index=1
        ),
        lx.data.Extraction(
            extraction_class="DNI",
            extraction_text="37.412.987",
            group_index=1
        ),
        # Direcciones de la víctima (ambas al grupo 1)
        lx.data.Extraction(
            extraction_class="DIRECCION",
            extraction_text="Av. Pedro Goyena 51, piso 7° dpto. \"A\"",
            attributes={"TIPO": "domicilio_de_la_victima"},
            group_index=1
        ),
        lx.data.Extraction(
            extraction_class="DIRECCION",
            extraction_text="Callao 543",
            attributes={"TIPO": "domicilio_de_la_victima"},
            group_index=1
        ),

        # Acusado / Imputado
        lx.data.Extraction(
            extraction_class="PER",
            extraction_text="ROBERTO CARUZO",
            attributes={"ROL_PROCESAL": "Acusado"},
            group_index=2
        ),
        lx.data.Extraction(
            extraction_class="DNI",
            extraction_text="30.112.642",
            group_index=2
        ),
        lx.data.Extraction(extraction_class="EDAD",        extraction_text="34", group_index=2),
        lx.data.Extraction(extraction_class="NACIONALIDAD",extraction_text="paraguaya", group_index=2),
        lx.data.Extraction(extraction_class="ESTUDIOS",    extraction_text="estudios secundarios completos", group_index=2),

        # Relación / Medidas (conecta grupos 2 -> 1)
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="cese en los actos de perturbación o intimidación por el plazo de seis meses",
            attributes={
                "TIPO": "cese_perturbacion",
                "SUJETO_ACTIVO_GROUP": 2,
                "SUJETO_PASIVO_GROUP": 1,
                "PLAZO": "6_meses"
            }
        ),
        lx.data.Extraction(
            extraction_class="RELACION",
            extraction_text="prohibición de acercamiento y contacto por el plazo de seis (6) meses a menos de 500 metros",
            attributes={
                "TIPO": "prohibicion_acercamiento",
                "SUJETO_ACTIVO_GROUP": 2,
                "SUJETO_PASIVO_GROUP": 1,
                "PLAZO": "6_meses",
                "DISTANCIA_MIN_M": 500,
                "LUGARES_ALCANZADOS": [
                    "Av. Pedro Goyena 51, piso 7° dpto. \"A\"",
                    "Callao 543",
                    "cualquier lugar donde se encuentre la víctima"
                ]
            }
        ),
    ],
    ),

    lx.data.ExampleData(
        text = textwrap.dedent(""""3) Abstenerse de ingresar y/o concurrir a la Villa 13"""),
        extractions = [
            lx.data.Extraction(extraction_class="LOC", extraction_text = "Villa 13")
        ]
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""JUZGADO NACIONAL EN LO CRIMINAL Y CORRECCIONAL N° 10 - Secretaría N° 19. "
              "Causa N° 52345/2022. CUIJ: 12-34567890-1. Actuación N° 2022-009876."""),
        extractions=[
            lx.data.Extraction(extraction_class="NUM_EXPEDIENTE", extraction_text="52345/2022"),
            lx.data.Extraction(extraction_class="CUIJ",           extraction_text="12-34567890-1"),
            lx.data.Extraction(extraction_class="NUM_ACTUACION",  extraction_text="2022-009876"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""Comparece Miguel Torres, DNI 30123456, de 34 años de edad, nacionalidad paraguaya, "
              "con estudios secundarios completos, con último domicilio en Av. Corrientes 3456 de esta ciudad, "
              "junto a su cuñado Jorge Pérez."""),
        extractions=[
            lx.data.Extraction(extraction_class="PER",         extraction_text="Miguel Torres"),
            lx.data.Extraction(extraction_class="DNI",         extraction_text="30123456"),
            lx.data.Extraction(extraction_class="EDAD",        extraction_text="34"),
            lx.data.Extraction(extraction_class="NACIONALIDAD",extraction_text="paraguaya"),
            lx.data.Extraction(extraction_class="ESTUDIOS",    extraction_text="estudios secundarios completos"),
            lx.data.Extraction(extraction_class="DIRECCION",   extraction_text="Av. Corrientes 3456"),
            lx.data.Extraction(extraction_class="PER",         extraction_text="Jorge Pérez"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""1) transferencia defondos a cuenta de terceros por $167.000,- hacia una cuenta a nombre de la Sra. Carla Analía Gonzales, CUIL 27-25011757-0, CBU 0740399088000036512321, del Banco Santander. """),
        extractions=[
            lx.data.Extraction(extraction_class="PER",         extraction_text="Carla Analía Gonzales"),
            lx.data.Extraction(extraction_class="CUIL",       extraction_text="27-25011757-0"),
            lx.data.Extraction(extraction_class="CBU",        extraction_text="0740399088000036512321"),
            lx.data.Extraction(extraction_class="BANCO",      extraction_text="Banco Santander"),
        ],
    ),

  lx.data.ExampleData(
        text=textwrap.dedent("""teléfono celular 1141504528 y dirección de correo electrónico alejandro.perezgarcia@gmail.com."""),
        extractions=[
            lx.data.Extraction(extraction_class="TELEFONO",     extraction_text="1141504528"),
            lx.data.Extraction(extraction_class="CORREO_ELECTRONICO", extraction_text="alejandro.perezgarcia@gmail.com"),
        ],
    ),
lx.data.ExampleData(
        text=textwrap.dedent("""Por otra parte, la División Investigaciones Judiciales de la Policía Federal Argentina informó que no se dio intervención a  ninguna otra Fiscalía u otro Juzgado por la sustracción del vehículo Volkswagen Voyage, dominio KXY-876 """),
   extractions=[
            lx.data.Extraction(extraction_class="PATENTE_DOMINIO", extraction_text="KXY-876"),
            lx.data.Extraction(extraction_class="MARCA_AUTOMOVIL", extraction_text="Volkswagen Voyage"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""Cecilia Lopez Gracia médica del Hospital Penna, Servicio SAME, M. N. 123.558."""),
        extractions=[
            lx.data.Extraction(extraction_class="PER", extraction_text="Cecilia Lopez Gracia"),
            lx.data.Extraction(extraction_class="NUM_MATRICULA", extraction_text="M. N. 123.558"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""A  su  vez,  requirió  informes  al  Banco BBVA   Francés, respecto  de  las  cuentas  bancarias  de  la  denunciante,  Carla Alejandra Garcia, D.N.I.  36.998.621 identificadas  como  Caja  de  ahorro  en  pesos  argentinos  número  117-59824/6  con  CBU  0180132640000004685591 y  Caja  de  ahorro  en  dólares  número  119-619018/2 con  CBU  0170115544000062081822."""),
        extractions=[
            lx.data.Extraction(extraction_class="PER",         extraction_text="Carla Alejandra Garcia"),
            lx.data.Extraction(extraction_class="DNI",         extraction_text="36.998.621"),
            lx.data.Extraction(extraction_class="NUM_CAJA_AHORRO", extraction_text="117-59824/6"),
            lx.data.Extraction(extraction_class="CBU",        extraction_text="0180132640000004685591"),
            lx.data.Extraction(extraction_class="NUM_CAJA_AHORRO", extraction_text="119-619018/2"),
            lx.data.Extraction(extraction_class="CBU",        extraction_text="0170115544000062081822"),
        ],
    ), 

    lx.data.ExampleData(
        text=textwrap.dedent("""La grabación se encuentra disponible en el link: https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv"""),
        extractions=[
            lx.data.Extraction(extraction_class="LINK", extraction_text="https://jusbairess.webex.com/jusbaire/njdndsgbnsdngsnv"),
        ],
    ),
    lx.data.ExampleData(
        text=textwrap.dedent("""VISTOS: Que a fin de ordenar la marcha del proceso, se fija audiencia preliminar. "
              "No se consignan números de expediente, CUIJ ni domicilios en el presente proveído."""),
        extractions=[],  # ejemplo negativo: desalienta devolver clases ausentes
    ),
]

SyntaxError: invalid syntax. Perhaps you forgot a comma? (608492658.py, line 20)

## Functions

In [16]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

def take_start_end_paragraphs(paragraphs):
    # Create start and end character positions
    start_end_chars = []
    current_pos = 0

    for i, paragraph in enumerate(paragraphs):
        start_char = current_pos
        end_char = start_char + len(paragraph)
        start_end_chars.append(
            {
                "paragraph_position": i,
                "text": paragraph,
                "paragraph_id": str(text_to_uuid(paragraph)).replace("-", ""),
                "start_char": start_char,
                "end_char": end_char,
            }
        )
        # +1 for the newline character between paragraphs (except after the last one)
        current_pos = end_char + 1

    return start_end_chars

def constract_paragraph(document):
    paragraphs = [line.strip() for line in document.split("\n") if line.strip()]
    paragraphs = [re.sub(r"\s{2,}", " ", line) for line in paragraphs]
    paragraphs = list(unique_justseen(paragraphs))
    return paragraphs

def langextract_to_dict(result):
    out = []
    for i in range(len(result.extractions)):
        #paragraph = result.text
        label = result.extractions[i].extraction_class
        text = result.extractions[i].extraction_text
        start_char = result.extractions[i].char_interval.start_pos
        end_char = result.extractions[i].char_interval.end_pos
        attrs = result.extractions[i].attributes
        alignment_status = result.extractions[i].alignment_status

        out.append({
            "label": label,
            "text": text,
            "start_char": start_char,
            "end_char": end_char,
            "attrs": attrs,
            "alignment_status": alignment_status
        })
        #print(f"{i+1}: \n CLASS: {result.extractions[i].extraction_class} \n TEXT: {result.extractions[i].extraction_text} \n from_chars: {text[result.extractions[i].char_interval.start_pos:result.extractions[i].char_interval.end_pos]}")
    return out

def langextract_prediction(text,PROMPT, examples,openai_api_key):

    result =  lx.extract(
        text_or_documents=text,
        prompt_description=PROMPT,
        examples=examples,
        language_model_type=lx.inference.OpenAILanguageModel,
        model_id="gpt-4o",
        api_key=openai_api_key,
        max_char_buffer=1000,
        extraction_passes=1,
        max_workers=6,
        fence_output=True,
        use_schema_constraints=False, # https://github.com/google/langextract
        language_model_params={
            "temperature": 0.1,
            "top_p": 0.9,
            "max_tokens": 400,
            "timeout": 600,},
            debug=False)
    return result, langextract_to_dict(result)

def sample_cases(df, label, model="prediction", n=3):
    # nombre dinámico para el campo de predicción principal
    pred_key = "openai" if model == "prediction" else ("ner" if model == "NER_prediction" else "predmodel")

    def _norm_label_text(item):
        if not isinstance(item, dict):
            return None, ""
        attrs = item.get("attrs", {}) or {}
        lab   = attrs.get("aymurai_label") or item.get("label")
        txt   = attrs.get("aymurai_alt_text") or item.get("extraction_text") or item.get("text", "")
        return lab, txt

    cases = {"TP": [], "FP": [], "FN": []}

    for _, row in df.iterrows():
        # validation
        val_raw = eval(row["validation"]) if row["validation"] else []
        val_spans = [_norm_label_text(v) for v in val_raw if isinstance(v, dict)]
        val_spans = [(lab, txt) for lab, txt in val_spans if lab is not None]

        # pred principal (según 'model')
        if model == "NER_prediction":
            pred_raw = eval(row[model]) if row.get(model) else []
        else:
            pred_raw = row.get(model) if row.get(model) else []
        pred_spans = [_norm_label_text(p) for p in pred_raw if isinstance(p, dict)]
        pred_spans = [(lab, txt) for lab, txt in pred_spans if lab is not None]

        # pred de ambos modelos (si existen las columnas)
        openai_raw = row.get("prediction")
        ner_raw    = row.get("NER_prediction")
        openai_list = openai_raw if (openai_raw and not isinstance(openai_raw, str)) else (eval(openai_raw) if openai_raw else [])
        ner_list    = ner_raw if (ner_raw and not isinstance(ner_raw, str)) else (eval(ner_raw) if ner_raw else [])

        openai_spans = [_norm_label_text(p) for p in (openai_list or []) if isinstance(p, dict)]
        openai_spans = [(lab, txt) for lab, txt in openai_spans if lab is not None]
        ner_spans    = [_norm_label_text(p) for p in (ner_list or []) if isinstance(p, dict)]
        ner_spans    = [(lab, txt) for lab, txt in ner_spans if lab is not None]

        # listas planas por label
        val_labels  = [lab for lab, _ in val_spans]
        pred_labels = [lab for lab, _ in pred_spans]

        # snippet e info extra
        text     = (row.get("text_x") or "")#[:300]
        para_id  = row.get("paragraph_id", None)
        doc_name = row.get("name", None)

        # armar registro con val + ambas preds + clave dinámica
        base_rec = {
            "paragraph_id": para_id,
            "document": doc_name,
            "parrafo": text,
            "val":   [t for lab, t in val_spans if lab == label],
            "openai": [t for lab, t in openai_spans if lab == label],
            "ner":    [t for lab, t in ner_spans    if lab == label],
        }
        # setear pred principal bajo su clave dinámica
        base_rec[pred_key] = [t for lab, t in pred_spans if lab == label]

        # clasificar TP/FP/FN
        if (label in val_labels) and (label in pred_labels):
            cases["TP"].append(base_rec)
        elif (label in val_labels) and (label not in pred_labels):
            cases["FN"].append(base_rec)
        elif (label not in val_labels) and (label in pred_labels):
            cases["FP"].append(base_rec)

    # muestra aleatoria top-n por tipo
    return {k: random.sample(v, min(len(v), n)) for k, v in cases.items() if v}


In [ ]:
docs2analize = ['2','3','4','6','7','8']
docs_file = [f for f in os.listdir(DOCS_PATH) if ('.docx' in f) and (len(set(docs2analize)&set(f))==1) ]

docs_file
documents = {}
doc_paragraphs = {}
doc_start_end_chars = {}
joined_texts = {}
for d in docs_file:
    path = DOCS_PATH + d
    # Extract document
    document = extraction(path)
    # Construct paragraphs
    paragraphs = constract_paragraph(document)
    start_end_chars = take_start_end_paragraphs(paragraphs)
    documents[d] = document
    doc_paragraphs[d] = paragraphs
    joined_text = "\n".join(paragraphs)
    joined_texts[d] = joined_text
    doc_start_end_chars[d] = start_end_chars


In [ ]:
paragraphs

In [20]:
sample_cases(df,'PER','NER_prediction')

{'TP': [{'paragraph_id': '50d23a7239aa5f088b2ceefac722563b',
   'document': 'document-03.docx',
   'parrafo': 'Asimismo, corresponde notificarlo de su derecho de designar defensa particular u oficial (arts. 28 y 29 CPP). Sin perjuicio de ello, a fin de garantizar el derecho de defensa en juicio del señor LUCAS GOMEZ frente a las medidas dispuestas, corresponde dar intervención a la Defensoría Oficial en turno con el objeto de notificarle lo resuelto. Dicha circunstancia será informada al acusado, poniendo a su disposición los datos de dicha dependencia.',
   'val': ['LUCAS GOMEZ'],
   'openai': [],
   'ner': ['LUCAS GOMEZ']},
  {'paragraph_id': 'de63b9abbe4c52ba877adf39d18b94d9',
   'document': 'document-04.docx',
   'parrafo': '3. DECLARAR REINCIDENTE al señor ELVIS JUNIOR GOMEZ, DNI n° 44.986.516 (art. 50 CP).',
   'val': ['ELVIS JUNIOR GOMEZ'],
   'openai': [],
   'ner': ['ELVIS JUNIOR GOMEZ']},
  {'paragraph_id': '00a10b79812d560e9e9a0ebfe5cf29ef',
   'document': 'document-04.docx'

In [ ]:
import time

start = timeit.timeit()
predictions = {}
results = {}

for name_doc, text in joined_texts.items():
    print(name_doc)
    results[name_doc], predictions[name_doc] = langextract_prediction(text, PROMPT, examples, openai_api_key)
    time.sleep(25)  # Sleep for 2 seconds to avoid rate limit

end = timeit.timeit()
print('\n Total time: ', end-start)
